# GT Change Mask: Why a Threshold? Can WorldNormal Help?

**Focus:** Three questions the data can answer without running any depth model.

1. **Why can't `depth_edit != depth_orig` give a perfect mask?** — UE float jitter
2. **What is the natural safe threshold?** — noise plateau auto-detection
3. **Does WorldNormal improve the depth mask?** — contact edges + thin-object simulation

Change `DATASET` in the Setup cell to switch between `new0` and `new1`.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import OpenEXR
import Imath

# -- Locate project root (works wherever Jupyter is launched from) --
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, 'data')):
    PROJECT_ROOT = _cwd
elif os.path.isdir(os.path.join(os.path.dirname(_cwd), 'data')):
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    raise RuntimeError('Cannot find project root. Launch Jupyter from UE_depth/ or change_detection_results/.')

NOTEBOOK_DIR = os.path.join(PROJECT_ROOT, 'change_detection_results')
GT_TO_METERS = 10000.0 / 100.0

# -- Dataset and thresholds --
DATASET       = 'new0'   # change to 'new1' to cross-check
DEPTH_THRESH  = 0.05     # metres  (current baseline in compare_edit_depth2.py)
NORMAL_THRESH = 5.0      # degrees

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATASET      : {DATASET}')

In [ ]:
def _exr_open(path):
    exr = OpenEXR.InputFile(path)
    dw  = exr.header()['dataWindow']
    w   = dw.max.x - dw.min.x + 1
    h   = dw.max.y - dw.min.y + 1
    return exr, h, w

def load_scene_depth(path):
    exr, h, w = _exr_open(path)
    FLOAT = Imath.PixelType(Imath.PixelType.FLOAT)
    for chan in ['R', 'SceneDepth', 'Z']:
        if chan in exr.header()['channels']:
            buf = exr.channel(chan, FLOAT)
            return np.frombuffer(buf, dtype=np.float32).reshape(h, w).copy() * GT_TO_METERS
    raise ValueError(f'No depth channel in {path}')

def load_world_normal(path):
    exr, h, w = _exr_open(path)
    FLOAT = Imath.PixelType(Imath.PixelType.FLOAT)
    ch  = [np.frombuffer(exr.channel(c, FLOAT), dtype=np.float32).reshape(h, w).copy()
           for c in 'RGB']
    raw = np.stack(ch, axis=-1)
    N   = raw / np.pi - 1.0
    mag = np.linalg.norm(N, axis=-1, keepdims=True)
    return N / (mag + 1e-6)

def load_rgb(path):
    return np.array(Image.open(path).convert('RGB'))

def angular_error_deg(n1, n2):
    dot = np.clip((n1 * n2).sum(axis=-1), -1.0, 1.0)
    return np.degrees(np.arccos(dot))

print('Helpers loaded.')

In [ ]:
folder = os.path.join(PROJECT_ROOT, 'data', DATASET)
files  = sorted(os.listdir(folder))

depth_exrs  = sorted([f for f in files if 'SceneDepth'  in f and 'WorldUnits' not in f and f.endswith('.exr')])
normal_exrs = sorted([f for f in files if 'WorldNormal' in f and f.endswith('.exr')])
rgb_pngs    = sorted([f for f in files if f.endswith('.png')])

depth_orig  = load_scene_depth(os.path.join(folder, depth_exrs[0]))
depth_edit  = load_scene_depth(os.path.join(folder, depth_exrs[1]))
normal_orig = load_world_normal(os.path.join(folder, normal_exrs[0]))
normal_edit = load_world_normal(os.path.join(folder, normal_exrs[1]))
rgb_edit    = load_rgb(os.path.join(folder, rgb_pngs[1]))

H, W        = depth_orig.shape
sky_mask    = (depth_orig > 500) | (depth_edit > 500)
non_sky     = ~sky_mask

depth_diff  = np.abs(depth_edit  - depth_orig)
normal_diff = angular_error_deg(normal_orig, normal_edit)

m_depth   = (depth_diff  > DEPTH_THRESH)  & non_sky
m_normal  = (normal_diff > NORMAL_THRESH) & non_sky
m_d_or_n  = m_depth | m_normal

print(f'Dataset : {DATASET}  ({W}x{H})')
print(f'Depth EXRs  : {depth_exrs}')
print(f'Normal EXRs : {normal_exrs}')
print(f'Sky pixels  : {sky_mask.mean()*100:.1f}%')
print(f'Changed (depth>{DEPTH_THRESH}m) : {m_depth.mean()*100:.1f}%')

---
## Q1 — Why can't `depth_edit != depth_orig` give a perfect mask?

UE renders the same static geometry twice. Ideally, unchanged pixels would have
**exactly the same depth value** — making `!= 0` a perfect binary detector.

In practice, UE's SceneDepth is a float32 Z-buffer that **jitters slightly between renders**
even for static geometry. The cells below quantify that jitter.

In [ ]:
vals    = depth_diff[non_sky]
nonzero = vals[vals > 0]

print('UE float jitter characterisation')
print('-' * 44)
print(f'  % pixels with ANY non-zero diff  : {(vals > 0).mean()*100:.1f}%')
print(f'  % pixels with diff > 0.1 mm      : {(vals > 0.0001).mean()*100:.1f}%')
print(f'  % pixels with diff > 1 mm        : {(vals > 0.001).mean()*100:.1f}%')
print(f'  % pixels with diff > 1 cm        : {(vals > 0.01).mean()*100:.1f}%')
print(f'  % pixels with diff > 2 cm        : {(vals > 0.02).mean()*100:.1f}%')
print(f'  % pixels with diff > 5 cm (THRESH): {(vals > 0.05).mean()*100:.1f}%')
print()
print(f'  Smallest non-zero diff : {nonzero.min()*100:.5f} cm')
print(f'  Median diff (all px)   : {np.median(vals)*100:.4f} cm')
print()
print(f'  => "!= 0" flags {(vals > 0).mean()*100:.0f}% as changed.')
print(f'     DEPTH_THRESH={DEPTH_THRESH}m flags only {(vals > DEPTH_THRESH).mean()*100:.1f}%.')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

log_bins = np.logspace(np.log10(nonzero.min()), np.log10(0.6), 200)
axes[0].hist(nonzero, bins=log_bins, color='steelblue', alpha=0.8, log=True)
axes[0].set_xscale('log')
axes[0].axvline(DEPTH_THRESH, color='red',    ls='--', lw=2, label=f'DEPTH_THRESH = {DEPTH_THRESH} m')
axes[0].axvline(0.02,         color='orange', ls='--', lw=1.5, label='0.02 m (approx noise cliff)')
axes[0].set_xlabel('|depth_edit - depth_orig|  (m,  log scale)')
axes[0].set_ylabel('Pixel count  (log)')
axes[0].set_title(f'Depth diff distribution -- {DATASET}\nLog-log shows noise cliff clearly')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, which='both')

thresholds  = np.linspace(0, 0.30, 300)
pct_flagged = [(vals > t).mean() * 100 for t in thresholds]
axes[1].plot(thresholds * 100, pct_flagged, 'b-', lw=2)
axes[1].axvline(DEPTH_THRESH * 100, color='red',    ls='--', lw=2, label=f'current {DEPTH_THRESH*100:.0f} cm')
axes[1].axvline(2.0,                color='orange', ls='--', lw=1.5, label='2 cm')
axes[1].set_xlabel('Depth threshold (cm)')
axes[1].set_ylabel('% pixels flagged changed')
axes[1].set_title('Cumulative % changed vs threshold\nLook for the plateau = pure noise region')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 30)

plt.suptitle(f'Q1: UE float jitter -- {DATASET}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Q2 — What is the natural safe threshold?

The cumulative plot above should show a **plateau**: a flat region where increasing the
threshold barely changes the % flagged. That flat region = jitter only. When the curve
starts rising steeply again, you're into real geometry changes.

**Auto-detection:** find where the derivative `d(% changed)/d(threshold)` drops to a
near-minimum (plateau), then rises again (true change signal).

In [ ]:
thresholds_fine = np.linspace(0.001, 0.20, 500)
pct_fine = np.array([(vals > t).mean() * 100 for t in thresholds_fine])
dpct     = np.abs(np.gradient(pct_fine, thresholds_fine))

# Plateau = derivative below 5% of its peak
plateau_mask = dpct < 0.05 * dpct.max()
plateau_idx  = np.where(plateau_mask)[0]

# Noise floor end = last index before derivative rises back for the final significant stretch
noise_floor_t = thresholds_fine[0]  # fallback
if len(plateau_idx) > 5:
    gaps      = np.diff(plateau_idx)
    big_gaps  = np.where(gaps > 8)[0]
    if len(big_gaps) > 0:
        noise_floor_t = thresholds_fine[plateau_idx[big_gaps[0]]]
    else:
        noise_floor_t = thresholds_fine[plateau_idx[-1]]

print(f'Auto-detected noise floor end : {noise_floor_t*100:.1f} cm  ({noise_floor_t:.3f} m)')
print(f'Current DEPTH_THRESH          : {DEPTH_THRESH*100:.0f} cm')
print(f'Safety margin                 : {(DEPTH_THRESH - noise_floor_t)*100:.1f} cm above noise floor')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(thresholds_fine * 100, pct_fine, 'b-', lw=2)
axes[0].axvline(noise_floor_t * 100, color='green', ls='--', lw=2,
                label=f'auto: noise floor end ({noise_floor_t*100:.1f} cm)')
axes[0].axvline(DEPTH_THRESH  * 100, color='red',   ls='--', lw=2,
                label=f'current threshold ({DEPTH_THRESH*100:.0f} cm)')
axes[0].set_xlabel('Depth threshold (cm)')
axes[0].set_ylabel('% pixels flagged changed')
axes[0].set_title('Cumulative % changed (zoomed in)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 20)

axes[1].plot(thresholds_fine * 100, dpct, 'purple', lw=2)
axes[1].axvline(noise_floor_t * 100, color='green', ls='--', lw=2,
                label=f'noise floor end ({noise_floor_t*100:.1f} cm)')
axes[1].axhline(0.05 * dpct.max(), color='gray', ls=':', lw=1.5,
                label='5% of peak (plateau criterion)')
axes[1].set_xlabel('Depth threshold (cm)')
axes[1].set_ylabel('|d(% changed) / d(threshold)|')
axes[1].set_title('Derivative -- flat = noise plateau')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 20)

plt.suptitle(f'Q2: Natural threshold detection -- {DATASET}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Q3 — Does WorldNormal improve the depth mask?

The depth mask is already visually good. Two scenarios where it might miss pixels:

1. **Contact edges** — where an added object sits on the floor or against a wall.
   At the exact contact point, the depth diff is small (object touches original surface),
   but the surface normal flips from floor to object face. Normal diff can catch these extra pixels.

2. **Thin/flat objects** (e.g. a poster on a wall) — depth diff = poster thickness.
   If that's below the noise floor (~2 cm), depth diff misses it entirely.
   The face of the poster has the **same normal as the wall**, so normal diff = 0 on the face.
   Only the edges of the poster have a normal change.

**Bottom line:** Normal helps at contact edges. It cannot rescue a flat-face thin object.

In [ ]:
depth_thresholds = [0.02, 0.05, 0.10]

fig, axes = plt.subplots(2, 3, figsize=(21, 10))
plt.subplots_adjust(hspace=0.06, wspace=0.04)

for col, dt in enumerate(depth_thresholds):
    md_only = (depth_diff > dt) & non_sky
    md_or_n = md_only | m_normal

    for row, (mask, label) in enumerate([
        (md_only, f'Depth only  (>{dt}m)  {md_only.mean()*100:.1f}%'),
        (md_or_n, f'Depth OR Normal  (d>{dt}m OR n>{NORMAL_THRESH}deg)  {md_or_n.mean()*100:.1f}%'),
    ]):
        ax = axes[row, col]
        ax.imshow(rgb_edit, alpha=0.45)
        ov = np.zeros((*mask.shape, 4), dtype=np.float32)
        ov[mask]               = [1.0, 0.15, 0.15, 0.75]
        ov[~mask & ~sky_mask]  = [0.0, 0.60, 0.00, 0.10]
        ax.imshow(ov)
        ax.set_title(label, fontsize=9)
        ax.axis('off')

plt.suptitle(
    f'Q3: Depth-only vs Depth OR Normal -- {DATASET}\n'
    'Red = changed  |  faint green = unchanged  |  white = sky',
    fontsize=13, fontweight='bold')
plt.savefig(os.path.join(NOTEBOOK_DIR, f'depth_vs_normal_grid_{DATASET}.png'),
            dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
ax.imshow(rgb_edit, alpha=0.4)

ov = np.zeros((H, W, 4), dtype=np.float32)
ov[m_depth &  m_normal] = [1.00, 0.55, 0.00, 0.85]   # orange = both agree
ov[m_depth & ~m_normal] = [0.90, 0.10, 0.10, 0.80]   # red    = depth only
ov[~m_depth & m_normal] = [0.10, 0.20, 0.95, 0.80]   # blue   = normal only
ax.imshow(ov)

both_pct        = (m_depth &  m_normal).mean() * 100
depth_only_pct  = (m_depth & ~m_normal).mean() * 100
normal_only_pct = (~m_depth & m_normal).mean() * 100

patches = [
    mpatches.Patch(color=[1,.55,0],   label=f'Both agree    ({both_pct:.1f}%)'),
    mpatches.Patch(color=[.9,.1,.1],  label=f'Depth only    ({depth_only_pct:.1f}%)'),
    mpatches.Patch(color=[.1,.2,.95], label=f'Normal only   ({normal_only_pct:.1f}%) -- inspect: contact edge or noise?'),
]
ax.legend(handles=patches, loc='lower right', fontsize=11)
ax.axis('off')
ax.set_title(
    f'3-way agreement -- {DATASET}\n'
    f'depth>{DEPTH_THRESH}m  |  normal>{NORMAL_THRESH} deg\n'
    'Blue = what WorldNormal adds over depth.  Zoom into blue to judge: real contact edges or noise?',
    fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, f'threeway_agreement_{DATASET}.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Normal-only pixels: {normal_only_pct:.2f}% of all non-sky pixels.')
print('These are the pixels WorldNormal would add to the depth mask.')

In [ ]:
def fit_scale_shift(pred, gt, unchanged):
    valid = unchanged & (gt > 0.1) & (gt < 100) & np.isfinite(pred) & np.isfinite(gt)
    if valid.sum() < 10:
        return float('nan'), float('nan'), 0
    p, g  = pred[valid].ravel(), gt[valid].ravel()
    A     = np.vstack([p, np.ones_like(p)]).T
    scale, shift = np.linalg.lstsq(A, g, rcond=None)[0]
    return float(scale), float(shift), int(valid.sum())

print('Calibration impact (depth_orig as pred, depth_edit as GT)')
print('Perfect predictor => scale=1.00, shift=0.00')
print()
print(f'  {"Mask":<30} {"scale":>8} {"shift(m)":>10} {"n_px":>10} {"|scale-1|":>10}')
print('  ' + '-' * 72)

mask_variants = [
    (~sky_mask,                              'No mask (all pixels)'),
    (~m_depth & ~sky_mask,                   f'Depth only  (d>{DEPTH_THRESH}m)'),
    (~m_d_or_n & ~sky_mask,                  f'Depth OR Normal'),
    (~(m_depth & m_normal) & ~sky_mask,      'Depth AND Normal'),
]
for unchanged, name in mask_variants:
    s, sh, n = fit_scale_shift(depth_orig, depth_edit, unchanged)
    err = abs(s - 1.0) if not np.isnan(s) else float('nan')
    print(f'  {name:<30} {s:>8.4f} {sh:>10.4f} {n:>10,} {err:>10.4f}')

print()
print('Lower |scale-1| = fewer contaminated (changed) pixels in the calibration fit.')

---
## Thin object simulation — can either signal detect a poster?

A poster on a wall:
- **Depth diff on face** = poster thickness (e.g. 0.5–5 cm). Below the noise floor → invisible.
- **Normal diff on face** = ~0 deg. A flat face parallel to the wall has the **same normal as the wall**,
  so both renders produce the same WorldNormal value. Normal diff cannot help here.
- **Normal diff on edges** = ~90 deg. But edges are a tiny fraction of the poster's pixel area.

The simulation below uses the **actual noise distribution** from the dataset to estimate the
probability of detecting an object of a given thickness.

In [ ]:
# Noise samples from pixels the depth mask considers unchanged
noise_samples = depth_diff[non_sky & ~m_depth].ravel()

print(f'Noise characterisation (unchanged pixels by depth mask):')
print(f'  n = {len(noise_samples):,}  |  mean = {noise_samples.mean()*100:.3f} cm  |  std = {noise_samples.std()*100:.3f} cm')
print(f'  p95 = {np.percentile(noise_samples, 95)*100:.3f} cm  |  p99 = {np.percentile(noise_samples, 99)*100:.3f} cm')
print()

# For a flat-face object of thickness T placed in front of the wall:
#   observed depth_diff = T + noise_sample  (object is closer by T)
#   detected if T + noise_sample > DEPTH_THRESH
thicknesses = np.linspace(0, 0.12, 200)  # 0 to 12 cm

p_at_5cm  = np.array([(noise_samples + t > DEPTH_THRESH).mean()         for t in thicknesses])
p_at_auto = np.array([(noise_samples + t > noise_floor_t).mean()        for t in thicknesses])
p_at_2cm  = np.array([(noise_samples + t > 0.02).mean()                 for t in thicknesses])

# Edge-only detection: what fraction of a square object is at the edges?
obj_widths_px = np.array([10, 20, 40, 80, 160, 320])
edge_fractions = np.minimum(4.0 / obj_widths_px, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(thicknesses * 100, p_at_5cm  * 100, 'b-',  lw=2, label=f'Threshold = {DEPTH_THRESH*100:.0f} cm (current)')
axes[0].plot(thicknesses * 100, p_at_auto * 100, 'g--', lw=2, label=f'Threshold = {noise_floor_t*100:.1f} cm (auto noise floor)')
axes[0].plot(thicknesses * 100, p_at_2cm  * 100, 'r:',  lw=2, label='Threshold = 2 cm')
axes[0].axhline(50, color='gray', ls=':', lw=1, label='50% detection')
axes[0].axhline(95, color='gray', ls=':', lw=1.5, label='95% detection')
axes[0].axvline(DEPTH_THRESH * 100,  color='blue',  ls=':', alpha=0.5)
axes[0].axvline(noise_floor_t * 100, color='green', ls=':', alpha=0.5)
axes[0].set_xlabel('Object thickness (cm)')
axes[0].set_ylabel('% of face pixels detected')
axes[0].set_title('Flat-face object detection probability\n(using empirical noise from unchanged pixels)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 12)

axes[1].bar([str(w) for w in obj_widths_px], edge_fractions * 100,
            color='steelblue', alpha=0.8, edgecolor='navy')
axes[1].axhline(1.0, color='red', ls='--', lw=2, label='1% edge fraction')
axes[1].set_xlabel('Object width (pixels)')
axes[1].set_ylabel('Edge pixel fraction (%)')
axes[1].set_title('Normal signal: fraction of object pixels\nwhere normal diff is detectable\n(edges only; face = 0 deg diff)')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(f'Thin object simulation -- {DATASET}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(NOTEBOOK_DIR, f'thin_object_simulation_{DATASET}.png'),
            dpi=130, bbox_inches='tight')
plt.show()

# 50% and 95% detection thicknesses
def thickness_at_pct(p_curve, target_pct):
    idx = np.searchsorted(p_curve, target_pct / 100.0)
    return thicknesses[idx] * 100 if idx < len(thicknesses) else float('nan')

print('Object thickness needed for reliable detection (current DEPTH_THRESH):')
print(f'  50% detection at : {thickness_at_pct(p_at_5cm, 50):.1f} cm')
print(f'  95% detection at : {thickness_at_pct(p_at_5cm, 95):.1f} cm')
print()
print('Normal can only add edge pixels (~1-2 px border) -- negligible for large flat objects.')
print('Conclusion: a poster thinner than ~{:.0f} cm on a wall is undetectable by either signal.'.format(
    thickness_at_pct(p_at_5cm, 50)))

---
## Summary

| Question | Answer |
|----------|--------|
| Why can't `depth != 0` work? | UE float jitter: ~80-84% of unchanged pixels differ by some non-zero amount |
| Minimum safe threshold | ~2 cm (auto-detected end of noise plateau) |
| Current threshold (5 cm) | Safe — well above noise floor, minimal false-positive risk |
| Does WorldNormal add pixels? | Yes — ~3% extra pixels at **contact edges** (object touching floor/wall) |
| Blue pixels on 3-way map | Inspect them: if they outline edges of the new object, they are real. If scattered across unchanged surfaces, they are noise. |
| Thin flat objects (poster) | Face: normal diff = 0 (same normal as wall). Depth diff = thickness. Neither signal helps below noise floor (~2 cm). |
| Recommendation | Use **Depth OR Normal** if contact edges matter for your scene. Use **Depth only** if the normal-only pixels look noisy. Never expect either signal to catch a flat-face object thinner than ~2 cm. |